# NeuroTCS v1.8.0 — Third-Party Reviewer Verification (Colab)

**Status:** Browser-only zero-install verification. Companion to the canonical [v2 protocol](https://github.com/DrMaruf1991/NeuroTCS/blob/main/docs/reviewer_package/reviewer_verification_prompt.md).

**Audience:** FDA technical staff, pharma diligence teams, peer reviewers, hospital AI governance committees wanting a fast preview without local installation.

**Runtime:** ~5 minutes on a standard Colab CPU instance.

## What this notebook proves

- The NeuroTCS framework installs cleanly from the public Apache-2.0 release
- All framework-internal tests pass (no cohort data required)
- The audit pipeline runs end-to-end on synthetic NIA-AA 2018 longitudinal data
- The audit_id is deterministic across runs

## What this notebook does NOT prove

Colab is third-party Google Cloud infrastructure. **DUA-controlled cohort data (NACC, ADNI, OASIS-3, MIRIAD) must not be uploaded here** — that would breach the DUAs and HIPAA. To verify the five locked audit invariants on real cohort data, run the [v2 protocol](https://github.com/DrMaruf1991/NeuroTCS/blob/main/docs/reviewer_package/reviewer_verification_prompt.md) on a local machine where you already have DUA-approved data on disk.

## Verdict this notebook can produce

`FRAMEWORK_INSTALL_VERIFIED` — equivalent to Steps 1–3 of the v2 protocol. A reviewer needing `FULL_REPRODUCED` must continue to the local v2 protocol with their own DUA data.

---
## Step 1 — Clone the locked release at v1.8.0

In [ ]:
!git clone --quiet --depth 1 --branch v1.8.0 https://github.com/DrMaruf1991/NeuroTCS.git 2>&1 | tail -3
%cd NeuroTCS
!git rev-parse HEAD
# EXPECTED: 9e8f693e3d5576e7c52507f4ee7f66699c1f6ce1

import subprocess
sha = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
EXPECTED_SHA = '9e8f693e3d5576e7c52507f4ee7f66699c1f6ce1'
assert sha == EXPECTED_SHA, f'Wrong release SHA: got {sha}, expected {EXPECTED_SHA}'
print(f'\n✓ Release SHA verified: {sha}')

## Step 2 — Verify Apache-2.0 license

In [ ]:
!head -2 LICENSE
print('✓ Apache-2.0 license confirmed' if 'Apache' in open('LICENSE').read()[:200] else '✗ LICENSE FILE UNEXPECTED')

## Step 3 — Install the framework

In [ ]:
!pip install -e . --quiet 2>&1 | tail -3

import neurotcs
print(f'\n✓ NeuroTCS version installed: {neurotcs.__version__}')
assert neurotcs.__version__ == '1.8.0', f'Version mismatch: {neurotcs.__version__}'

## Step 4 — Run framework-only tests (no cohort data required)

Expected: 401 passed. The 7 cohort regression tests skip because Colab has no access to DUA-controlled data.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/', '-q', '--no-header', '--tb=no'],
    capture_output=True, text=True, timeout=600,
)
print(result.stdout[-2000:])

# Parse the final line
final_line = [ln for ln in result.stdout.splitlines() if 'passed' in ln or 'failed' in ln]
final = final_line[-1] if final_line else ''
print(f'\nFinal: {final}')

# Hard assertion — no failures, at least 401 passed
import re
m_pass = re.search(r'(\d+) passed', final)
m_fail = re.search(r'(\d+) failed', final)
n_passed = int(m_pass.group(1)) if m_pass else 0
n_failed = int(m_fail.group(1)) if m_fail else 0

assert n_failed == 0, f'{n_failed} tests failed — framework not reliable on Colab'
assert n_passed >= 401, f'Only {n_passed} tests passed; expected ≥ 401'
print(f'\n✓ {n_passed} framework-only tests passed, 0 failed')

## Step 5 — Synthetic-data audit demonstration

Generates 200 synthetic NIA-AA 2018 longitudinal trajectories and runs the audit end-to-end. This proves the pipeline works without needing real cohort data.

**Important:** the synthetic audit_id below is NOT one of the five locked invariants — it's a demonstration that the pipeline runs. Locked invariants require real cohort data and are verified by the v2 protocol on a local machine.

In [ ]:
# Inline synthetic data generator — does not depend on any external file
import datetime as _dt
import hashlib as _hashlib
import random as _random
from neurotcs.audit_core import Trajectory

_STATES = ['CN', 'MCI', 'AD']
_ADMISSIBLE = {('CN','CN'),('CN','MCI'),('MCI','MCI'),('MCI','AD'),('MCI','CN'),('AD','AD')}

def _hash_id(idx, salt='colab_demo_v1'):
    return _hashlib.sha256(f'{salt}_{idx:06d}'.encode()).hexdigest()[:16]

def generate_synthetic(n_subjects=200, seed=42, n_implausible=3):
    rng = _random.Random(seed)
    base = _dt.date(2020, 1, 1)
    trajs = []
    for idx in range(n_subjects):
        n_visits = rng.randint(2, 5)
        start = rng.choices(_STATES, weights=[0.5, 0.4, 0.1], k=1)[0]
        states = [start]
        dates = [base + _dt.timedelta(days=rng.randint(0, 365))]
        for _ in range(n_visits - 1):
            curr = states[-1]
            nxts = [s for s in _STATES if (curr, s) in _ADMISSIBLE] or [curr]
            wts = [0.6 if n==curr else (0.3 if _STATES.index(n)>_STATES.index(curr) else 0.1) for n in nxts]
            states.append(rng.choices(nxts, weights=wts, k=1)[0])
            dates.append(dates[-1] + _dt.timedelta(days=rng.randint(180, 540)))
        trajs.append(Trajectory(patient_id=_hash_id(idx), states=tuple(states), dates=tuple(dates)))
    # seed implausible transitions for audit visibility
    for k in range(n_implausible):
        i = (k*37 + 11) % len(trajs)
        t = trajs[i]
        if len(t.states) >= 2:
            ns = list(t.states); ns[-1] = 'CN' if ns[0] != 'CN' else 'AD'
            trajs[i] = Trajectory(patient_id=t.patient_id, states=tuple(ns), dates=t.dates)
    return trajs

from neurotcs import audit, load_rulepack
rp = load_rulepack('ad/niaaa_2018')
trajs = generate_synthetic(n_subjects=200, seed=42)
print(f'Generated {len(trajs)} synthetic subjects, {sum(len(t.states)-1 for t in trajs)} transitions')

result = audit(trajs, rp, bootstrap_B=10_000, seed=42, ci_method='bca')
print(f'\nSynthetic audit result:')
print(f'  cTCS:           {result.ctcs.ci.point:.6f}')
print(f'  audit_id:       {result.audit_id}')
print(f'  Patients:       {result.n_patients_scored}')
print(f'  Transitions:    {result.n_transitions}')
print(f'  Flagged:        {result.n_flagged} (synthetic implausibles seeded for visibility)')

# Range check
in_range = 0.90 <= result.ctcs.ci.point <= 1.00
print(f'\n✓ cTCS in [0.90, 1.00]: {in_range}')
synthetic_audit_id = result.audit_id

## Step 6 — Determinism check (run the audit twice)

Same seed, same data, must produce byte-identical audit_id.

In [ ]:
trajs2 = generate_synthetic(n_subjects=200, seed=42)
result2 = audit(trajs2, rp, bootstrap_B=10_000, seed=42, ci_method='bca')

print(f'Run 1 audit_id: {synthetic_audit_id}')
print(f'Run 2 audit_id: {result2.audit_id}')

deterministic = (synthetic_audit_id == result2.audit_id)
print(f'\n✓ Byte-identical across runs: {deterministic}')
assert deterministic, 'Non-deterministic — audit_id changed between identical runs'

## Step 7 — Read the open-limitations disclosure

Reviewer must read this. Anything else is incomplete review.

In [ ]:
# Pull the limitations section from the datasheet
import re
datasheet = open('docs/datasheet/ad_neurotcs_datasheet.md').read()
# Find Limitations section
m = re.search(r'#+ .*[Ll]imitations.*?(?=\n#+ |\Z)', datasheet, re.DOTALL)
if m:
    print(m.group(0)[:3000])
else:
    print('No limitations section found — flag this as a concern')

## Step 8 — Emit YAML partial attestation

This produces a `FRAMEWORK_INSTALL_VERIFIED` attestation. Replace placeholder fields and save.

In [ ]:
import datetime, platform, sys
from datetime import timezone

import numpy as np, pandas as pd
try:
    import pyreadr; pyreadr_ver = pyreadr.__version__
except Exception:
    pyreadr_ver = 'not_installed'

attestation = f'''attestation:
  schema_version: "1.0"
  protocol_version: "v2-colab"
  protocol_url: "https://github.com/DrMaruf1991/NeuroTCS/blob/main/docs/reviewer_package/reviewer_verification_prompt.md"
  verified_release_tag: "v1.8.0"
  verified_release_commit: "{sha}"
  execution_surface: "google_colab"

reviewer:
  name: "<FILL_IN>"
  affiliation: "<FILL_IN>"
  email: "<FILL_IN>"
  orcid: "<OPTIONAL>"
  role: "<FDA_REVIEWER|PHARMA_DILIGENCE|ACADEMIC_PEER|HOSPITAL_AI_GOVERNANCE|INDEPENDENT_TECHNICAL>"

environment:
  date_utc: "{datetime.datetime.now(timezone.utc).isoformat()}"
  os: "{platform.platform()}"
  python_version: "{sys.version.split()[0]}"
  numpy_version: "{np.__version__}"
  pandas_version: "{pd.__version__}"
  pyreadr_version: "{pyreadr_ver}"

framework_tests:
  step_4_passed: {n_passed}
  step_4_failed: {n_failed}
  step_4_outcome: "PASSED"

synthetic_demonstration:
  n_subjects: 200
  n_transitions: {sum(len(t.states)-1 for t in trajs)}
  cTCS_observed: "{result.ctcs.ci.point:.6f}"
  audit_id_observed: "{synthetic_audit_id}"
  cTCS_in_clinical_range: {str(in_range).lower()}
  determinism_observed: {str(deterministic).lower()}

cohort_audit_results:
  note: "Not executable on Colab — DUA-controlled cohort data must not be
    uploaded to third-party cloud. Use v2 protocol on local machine for
    OASIS-3, ADNI, NACC, MIRIAD verification."

verdict:
  reproducibility_verdict: "FRAMEWORK_INSTALL_VERIFIED"
  verdict_rationale: |
    Framework cloned at locked v1.8.0 release commit, installed cleanly,
    {n_passed} framework-only tests passed with 0 failures. Synthetic-data
    audit pipeline executed end-to-end producing deterministic audit_id
    {synthetic_audit_id[:16]}... with cTCS in clinical range. Real-cohort
    verification not performed (Colab is unsuitable for DUA data).
  scope_limitation: |
    This attestation establishes ONLY that the framework installs and runs.
    It does NOT verify the five locked audit invariants on real cohort data.
    For FULL_REPRODUCED verdict, complete the v2 protocol on a local machine.

signature:
  reviewer_signature: "<FILL_IN_TYPED_NAME_OR_GPG>"
  signature_method: "<TYPED_NAME|GPG|S/MIME>"
  signature_date_utc: "<FILL_IN>"
'''

# Write to file and offer for download
with open('neurotcs_v1.8.0_colab_partial_attestation.yaml', 'w') as f:
    f.write(attestation)

print('=== ATTESTATION ===')
print(attestation)
print('=== END ===')
print('\n✓ Saved to: neurotcs_v1.8.0_colab_partial_attestation.yaml')
print('  (use File panel on left to download, or run the next cell)')

In [ ]:
# Download the attestation to your local machine
from google.colab import files
files.download('neurotcs_v1.8.0_colab_partial_attestation.yaml')

---
## Summary

What you verified on Colab today:

- ✓ Framework cloned at locked commit `9e8f693e`
- ✓ Apache-2.0 license confirmed
- ✓ 401 framework-only tests passed, 0 failed
- ✓ Synthetic-data audit produced deterministic audit_id
- ✓ cTCS in clinical range on synthetic data
- ✓ Limitations section read
- ✓ Partial YAML attestation generated

## What you should do next (if you want a stronger verdict)

1. **For `METHOD_CONSISTENT_DIFFERENT_FREEZE`:** run the v2 protocol on a local machine with **any** cohort data you have under DUA (just one cohort is enough for this verdict).
2. **For `FULL_REPRODUCED`:** run the v2 protocol on a local machine with cohort data whose SHA-256s match the manifest in `docs/reproducibility/cohort_input_checksums.md` of the release.
3. **For your records:** sign and submit the partial attestation downloaded above to the sponsor or to your committee/journal of record.

## Honest caveats

- This notebook does not replace per-aim study evidence (the substantive validation of NeuroTCS).
- A `FRAMEWORK_INSTALL_VERIFIED` verdict is informative but not sufficient for FDA clearance, peer-review acceptance, or hospital deployment.
- The synthetic audit_id is **not** one of the five locked invariants. It is a demonstration only.